In [0]:
from pyspark.sql.functions import regexp_replace, col, trim ,split , round , when , expr
import re 

# Path to your file
# file_path = "/Volumes/dbacademy/default/default_volume/Open Transaction Data.csv"
file_path = "/Volumes/dbacademy/default/datasets/Open Transaction DataFULL.csv"

# Read the file
df = (spark.read
      .option("header", "true")
      .option("sep", "\t")
      .option("encoding", "UTF-16")
      .csv(file_path))

# Clean column names by removing trailing spaces
for old_col in df.columns:
    new_col = old_col.strip()
    if old_col != new_col:
        df = df.withColumnRenamed(old_col, new_col)

# Remove all characters except letters and numbers
df = df.toDF(*[re.sub('[^0-9a-zA-Z]', '', col) for col in df.columns])

# Drop empty column if exists
if '' in df.columns or 'c13' in df.columns:
    df = df.drop('', 'c13')

 
# Clean and transform columns

df = df.select(
    "*",  # Keep all existing columns
    # Clean TransactionPrice: remove RM, commas, and hyphens, then handle empty strings
    when(regexp_replace(col("TransactionPrice"), "RM|,|-", "") == "", None)
        .otherwise(expr("try_cast(regexp_replace(TransactionPrice, 'RM|,|-', '') as double)"))
        .alias("TransactionPrice1"),
    # Clean MainFloorArea
    when(regexp_replace(col("MainFloorArea"), ",|-", "") == "", None)
        .otherwise( expr("try_cast(regexp_replace(MainFloorArea, ',|-', '') as double)"))
        .alias("MainFloorArea1"),
    # Clean LandParcelArea
    when(regexp_replace(col("LandParcelArea"), ",|-", "") == "", None)
        .otherwise(expr("try_cast(regexp_replace(LandParcelArea, ',|-', '') as double)"))
        .alias("LandParcelArea1"),
    # Clean UnitLevel
    expr("try_cast(trim(UnitLevel) as integer)").alias("UnitLevel1"),
    split(col("MonthYearofTransactionDate"), " ")[0].alias("month"),
    split(col("MonthYearofTransactionDate"), " ")[1].alias("year")
) 

#Derived columns
df = df.select(
    "*",
    when(col("MainFloorArea1") == 0, None)
        .otherwise(round(col("TransactionPrice1") / col("MainFloorArea1"), 2))
        .alias("transaction_price_per_main_floor_area"),
    when(col("LandParcelArea1") == 0, None)
        .otherwise(round(col("TransactionPrice1") / col("LandParcelArea1"), 2))
        .alias("transaction_price_per_land_parcel_area")
)
# Clean duplicate column name

# Get all column names
df_cols = df.columns
# Find indices of duplicate columns
duplicate_col_index = [idx for idx, val in enumerate(df_cols) if val in df_cols[:idx]]
# Create a copy of column names list
new_cols = df_cols.copy()
# Rename duplicate columns by adding suffix
for i in duplicate_col_index:
    if i>0:
        new_cols[i] = new_cols[i - 1] + new_cols[i]  
# Apply new column names to dataframe
df = df.toDF(*new_cols)

# Calculation and derived columns


# Display
display(df)
print(df.count())

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dbacademy.default.open_transaction_data_full")

In [0]:
df_silver = spark.sql ("""
SELECT 
year , month ,PropertyType, District , Mukim  ,  
AVG(transaction_price_per_main_floor_area) ,
AVG(transaction_price_per_land_parcel_area) ,
COUNT(*) as qty
FROM dbacademy.default.open_transaction_data_full
WHERE 
GROUP BY year , month, PropertyType, District , Mukim   
ORDER BY  year , month, PropertyType, District , Mukim   
""")

display(df_silver)